# Обучение VAE на FFHQ 128x128

Этот ноутбук обучает Variational Autoencoder (VAE) на датасете FFHQ с разрешением 128x128.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import os

In [ ]:
# Импорт модели VAE
import sys
sys.path.append('..')
from model import VAE, vae_loss

In [ ]:
# Гиперпараметры
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
LATENT_DIM = 256
NUM_EPOCHS = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Используемое устройство: {DEVICE}")

## Загрузка данных

In [ ]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class FFHQDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform or transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        
        self.image_paths = []
        for ext in ['.jpg', '.jpeg', '.png']:
            self.image_paths.extend(list(self.root_dir.glob(f'*{ext}')))
        
        print(f"Загружено {len(self.image_paths)} изображений")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image

# Создание DataLoader
data_dir = '../data/ffhq_128'
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

dataset = FFHQDataset(data_dir, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

## Инициализация модели

In [ ]:
model = VAE(latent_dim=LATENT_DIM).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Количество параметров: {sum(p.numel() for p in model.parameters()):,}")

## Функция обучения

In [ ]:
def train_epoch(model, dataloader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    total_recon_loss = 0
    total_kl_loss = 0
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch}')
    for batch_idx, data in enumerate(pbar):
        data = data.to(device)
        
        optimizer.zero_grad()
        recon, mu, logvar = model(data)
        loss, recon_loss, kl_loss = vae_loss(recon, data, mu, logvar, beta=1.0)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_kl_loss += kl_loss.item()
        
        if batch_idx % 10 == 0:
            pbar.set_postfix({
                'loss': f'{loss.item()/data.size(0):.4f}',
                'recon': f'{recon_loss.item()/data.size(0):.4f}',
                'kl': f'{kl_loss.item()/data.size(0):.4f}'
            })
    
    avg_loss = total_loss / len(dataset)
    avg_recon = total_recon_loss / len(dataset)
    avg_kl = total_kl_loss / len(dataset)
    
    return avg_loss, avg_recon, avg_kl

## Цикл обучения

In [ ]:
history = {'loss': [], 'recon_loss': [], 'kl_loss': []}

# Директория для сохранения чекпоинтов
checkpoint_dir = Path('./checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

for epoch in range(NUM_EPOCHS):
    avg_loss, avg_recon, avg_kl = train_epoch(
        model, dataloader, optimizer, DEVICE, epoch
    )
    
    history['loss'].append(avg_loss)
    history['recon_loss'].append(avg_recon)
    history['kl_loss'].append(avg_kl)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Loss: {avg_loss:.4f}, Recon: {avg_recon:.4f}, KL: {avg_kl:.4f}")
    
    # Сохранение чекпоинта каждые 10 эпох
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, checkpoint_dir / f'vae_checkpoint_epoch_{epoch+1}.pth')

# Сохранение финальной модели
torch.save(model.state_dict(), './vae_final.pth')
print("Обучение завершено!")

## Визуализация результатов обучения

In [ ]:
# График потерь
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Total Loss')
plt.plot(history['recon_loss'], label='Reconstruction Loss')
plt.plot(history['kl_loss'], label='KL Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Losses')

plt.subplot(1, 2, 2)
plt.plot(history['loss'], label='Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Total Loss')

plt.tight_layout()
plt.savefig('./training_loss.png', dpi=150)
plt.show()

## Генерация изображений (Инференс)

In [ ]:
def generate_samples(model, device, num_samples=16):
    model.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, LATENT_DIM).to(device)
        samples = model.decode(z)
        samples = samples.cpu()
    return samples

# Генерация образцов
samples = generate_samples(model, DEVICE, num_samples=16)

# Визуализация
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flatten()):
    img = samples[i].permute(1, 2, 0) * 0.5 + 0.5
    img = img.clamp(0, 1)
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.savefig('./generated_samples.png', dpi=150)
plt.show()

## Реконструкция изображений

In [ ]:
def reconstruct_samples(model, dataloader, device, num_samples=8):
    model.eval()
    with torch.no_grad():
        batch = next(iter(dataloader))
        batch = batch[:num_samples].to(device)
        recon, mu, logvar = model(batch)
    
    return batch.cpu(), recon.cpu()

# Реконструкция
original, reconstructed = reconstruct_samples(model, dataloader, DEVICE, num_samples=8)

# Визуализация
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

for i in range(8):
    img_orig = original[i].permute(1, 2, 0) * 0.5 + 0.5
    img_orig = img_orig.clamp(0, 1)
    axes[0, i].imshow(img_orig)
    axes[0, i].axis('off')
    
    img_recon = reconstructed[i].permute(1, 2, 0) * 0.5 + 0.5
    img_recon = img_recon.clamp(0, 1)
    axes[1, i].imshow(img_recon)
    axes[1, i].axis('off')

axes[0, 0].set_title('Original')
axes[1, 0].set_title('Reconstructed')
plt.tight_layout()
plt.savefig('./reconstruction.png', dpi=150)
plt.show()

## Интерполяция в латентном пространстве

In [ ]:
def interpolate_latent(model, device, num_steps=10):
    model.eval()
    with torch.no_grad():
        z1 = torch.randn(1, LATENT_DIM).to(device)
        z2 = torch.randn(1, LATENT_DIM).to(device)
        
        alphas = torch.linspace(0, 1, num_steps).view(-1, 1).to(device)
        z_interp = alphas * z1 + (1 - alphas) * z2
        
        samples = model.decode(z_interp).cpu()
    
    return samples

interp_samples = interpolate_latent(model, DEVICE, num_steps=10)

fig, axes = plt.subplots(2, 5, figsize=(15, 3))
for i, ax in enumerate(axes.flatten()):
    img = interp_samples[i].permute(1, 2, 0) * 0.5 + 0.5
    img = img.clamp(0, 1)
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.savefig('./interpolation.png', dpi=150)
plt.show()